# Phase 1 Kaggle - DeepSeek-OCR Only

Notebook n?y ch? test DeepSeek-OCR tr?n Kaggle: render PDF th?nh ?nh trang, ch?y DeepSeek-OCR tr?n GPU, r?i xu?t `book.md`, `rag_chunks.jsonl`, metadata theo page v? file zip. Kh?ng c?i MinerU, kh?ng EasyOCR, kh?ng Qwen spellcheck trong l??t n?y ?? c? l?p l?i.

In [ ]:
from pathlib import Path
import json, math, os, re, shutil, subprocess, sys, time, zipfile
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import datetime, timezone

# Kaggle c? th? preinstall TensorFlow/Flax; ?p Transformers ?i nh?nh PyTorch-only.
os.environ.setdefault('USE_TF', '0')
os.environ.setdefault('TRANSFORMERS_NO_TF', '1')
os.environ.setdefault('USE_FLAX', '0')
os.environ.setdefault('TRANSFORMERS_NO_FLAX', '1')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('PYTHONUTF8', '1')
os.environ.setdefault('PYTHONIOENCODING', 'utf-8')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

# Thay ???ng d?n input c?a file PDF v?o ??y.
INPUT_PDF = Path('/kaggle/input/vietnam-schoolbooks/SGK L?ch s? v? ??a l? 6 CD.pdf')

# N?u t?n dataset/file b? ??i tr?n Kaggle, t? t?m file PDF ??u ti?n.
if not INPUT_PDF.exists() and Path('/kaggle/input').exists():
    candidates = sorted(Path('/kaggle/input').rglob('*.pdf'))
    if candidates:
        INPUT_PDF = candidates[0]

# Fallback khi ch?y local trong repo.
if not INPUT_PDF.exists():
    INPUT_PDF = Path('books/SGK L?ch s? v? ??a l? 6 CD.pdf')

OUTPUT_DIR = Path('/kaggle/working/class_6_deepseek_only') if Path('/kaggle/working').exists() else Path('extracted/class_6_deepseek_only')
PAGES_DIR = OUTPUT_DIR / 'pages'
DEEPSEEK_OUT_DIR = OUTPUT_DIR / '_deepseek_pages'
METADATA_DIR = OUTPUT_DIR / 'metadata'
PREVIEW_DIR = OUTPUT_DIR / '_previews'
MANIFEST_DIR = OUTPUT_DIR / '_manifests'

# Test tr??c pages 6-10. Khi OK th? ??i START_PAGE=1, END_PAGE=None ?? ch?y to?n b? s?ch.
START_PAGE = 6      # 1-based
END_PAGE = 10       # None = ch?y h?t s?ch
RENDER_SCALE = 2.0

# DeepSeek-OCR ch?nh th?c d?ng model n?y. C? DeepSeek-OCR-2 m?i h?n, nh?ng notebook n?y ?u ti?n model ?ang c? infer() r? r?ng trong repo g?c.
DEEPSEEK_MODEL = 'deepseek-ai/DeepSeek-OCR'
DEEPSEEK_PROMPT = '<image>\n<|grounding|>Convert the document to markdown.'
DEEPSEEK_ATTN_IMPL = 'eager'  # Kaggle/T4 th??ng kh?ng ti?n c?i flash-attn; sdpa c? th? kh?ng ???c model h? tr?.
DEEPSEEK_BASE_SIZE = 1024
DEEPSEEK_IMAGE_SIZE = 640
DEEPSEEK_CROP_MODE = True
DEEPSEEK_TEST_COMPRESS = True

# Test ?n ??nh tr??c v?i 1 GPU. Sau khi pages 6-10 OK, c? th? ??i MAX_PARALLEL_GPUS=2 tr?n T4 x2 ?? chia trang.
GPU_IDS_TO_USE = 'auto'       # 'auto', '0', '0,1'
MAX_PARALLEL_GPUS = 1
ALLOW_CPU = False
FORCE_DEEPSEEK = False

CHUNK_SIZE = 1800
CHUNK_OVERLAP = 200
ZIP_OUTPUT = True

for directory in [OUTPUT_DIR, PAGES_DIR, DEEPSEEK_OUT_DIR, METADATA_DIR, PREVIEW_DIR, MANIFEST_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

print('INPUT_PDF =', INPUT_PDF)
print('OUTPUT_DIR =', OUTPUT_DIR)
print('Page range =', START_PAGE, END_PAGE)


In [ ]:
def run_cmd(cmd, *, env=None, check=True):
    print('+', ' '.join(map(str, cmd)))
    sys.stdout.flush()
    return subprocess.run(list(map(str, cmd)), env=env, check=check)

# Theo requirements.txt ch?nh th?c c?a deepseek-ai/DeepSeek-OCR:
# transformers==4.46.3 tokenizers==0.20.3 PyMuPDF img2pdf einops easydict addict Pillow numpy
# Kh?ng c?i torch/torchvision ? ??y ?? tr?nh pip k?o l?ch CUDA stack c?a Kaggle.
run_cmd([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'pip'])
run_cmd([
    sys.executable, '-m', 'pip', 'install', '-q', '--no-cache-dir',
    'transformers==4.46.3',
    'tokenizers==0.20.3',
    'PyMuPDF',
    'img2pdf',
    'einops',
    'easydict',
    'addict',
    'Pillow',
    'numpy<2.5',
    'accelerate',
    'safetensors',
    'python-docx',
    'tqdm',
    'matplotlib',
])

try:
    import torch
    GPU_COUNT = torch.cuda.device_count()
    GPU_IDS = list(range(GPU_COUNT))
except Exception as exc:
    print('torch import failed:', repr(exc))
    GPU_COUNT = 0
    GPU_IDS = []

CPU_COUNT = os.cpu_count() or 2
os.environ['OMP_NUM_THREADS'] = str(CPU_COUNT)
os.environ['MKL_NUM_THREADS'] = str(CPU_COUNT)
print('CPU_COUNT =', CPU_COUNT)
print('GPU_IDS =', GPU_IDS)

if not GPU_IDS and not ALLOW_CPU:
    raise RuntimeError('No CUDA GPU found. B?t GPU trong Kaggle Notebook Settings tr??c khi ch?y DeepSeek-OCR.')


In [ ]:
import torch
print('torch =', torch.__version__)
print('cuda available =', torch.cuda.is_available())
if torch.cuda.is_available():
    for idx in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(idx)
        free_bytes, total_bytes = torch.cuda.mem_get_info(idx)
        print(f'GPU {idx}: {props.name}, total={total_bytes/1024**3:.2f}GB, free={free_bytes/1024**3:.2f}GB')

# In th?m nvidia-smi n?u c?, ?? ph?t hi?n process kh?c chi?m VRAM.
run_cmd(['nvidia-smi'], check=False)


In [ ]:
import fitz
from tqdm.auto import tqdm


def render_one_page(args):
    pdf_path, page_index, scale, out_dir = args
    doc = fitz.open(str(pdf_path))
    page_no = page_index + 1
    out_path = Path(out_dir) / f'page_{page_no:03d}.jpg'
    if not out_path.exists():
        pix = doc[page_index].get_pixmap(matrix=fitz.Matrix(scale, scale), alpha=False)
        pix.save(str(out_path))
    return {'page_number': page_no, 'path': str(out_path)}

pdf_doc = fitz.open(str(INPUT_PDF))
total_pages = len(pdf_doc)
start_idx = max(0, START_PAGE - 1)
end_idx = total_pages if END_PAGE is None else min(total_pages, END_PAGE)
page_indexes = list(range(start_idx, end_idx))
print('total_pages =', total_pages, 'selected =', len(page_indexes))

render_jobs = [(INPUT_PDF, i, RENDER_SCALE, PAGES_DIR) for i in page_indexes]
page_images = []
workers = min(CPU_COUNT, max(1, len(render_jobs)))
with ProcessPoolExecutor(max_workers=workers) as executor:
    futures = [executor.submit(render_one_page, job) for job in render_jobs]
    for future in tqdm(as_completed(futures), total=len(futures), desc='Render PDF pages'):
        page_images.append(future.result())
page_images = sorted(page_images, key=lambda item: item['page_number'])
print('rendered pages:', len(page_images))
print(page_images[:3])


In [ ]:
WORKER_PATH = OUTPUT_DIR / 'deepseek_ocr_worker.py'
worker_py = r'''
import json
import os
import sys
import traceback
from pathlib import Path

os.environ.setdefault('USE_TF', '0')
os.environ.setdefault('TRANSFORMERS_NO_TF', '1')
os.environ.setdefault('USE_FLAX', '0')
os.environ.setdefault('TRANSFORMERS_NO_FLAX', '1')
os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

import torch
from transformers import AutoModel, AutoTokenizer

manifest_path = Path(sys.argv[1])
out_dir = Path(sys.argv[2])
out_dir.mkdir(parents=True, exist_ok=True)

model_name = os.environ.get('DEEPSEEK_MODEL', 'deepseek-ai/DeepSeek-OCR')
prompt = os.environ.get('DEEPSEEK_PROMPT', '<image>\n<|grounding|>Convert the document to markdown.')
attn_impl = os.environ.get('DEEPSEEK_ATTN_IMPL', 'eager')
base_size = int(os.environ.get('DEEPSEEK_BASE_SIZE', '1024'))
image_size = int(os.environ.get('DEEPSEEK_IMAGE_SIZE', '640'))
crop_mode = os.environ.get('DEEPSEEK_CROP_MODE', '1') == '1'
test_compress = os.environ.get('DEEPSEEK_TEST_COMPRESS', '1') == '1'
force = os.environ.get('FORCE_DEEPSEEK', '0') == '1'

print('worker cuda visible =', os.environ.get('CUDA_VISIBLE_DEVICES'))
print('torch =', torch.__version__, 'cuda =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('device =', torch.cuda.get_device_name(0))
    free, total = torch.cuda.mem_get_info(0)
    print(f'gpu memory before load: free={free/1024**3:.2f}GB total={total/1024**3:.2f}GB')

records = [json.loads(line) for line in manifest_path.read_text(encoding='utf-8').splitlines() if line.strip()]

tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
load_kwargs = {
    'trust_remote_code': True,
    'use_safetensors': True,
    '_attn_implementation': attn_impl,
    'low_cpu_mem_usage': True,
}
try:
    model = AutoModel.from_pretrained(model_name, **load_kwargs)
except TypeError:
    load_kwargs.pop('low_cpu_mem_usage', None)
    model = AutoModel.from_pretrained(model_name, **load_kwargs)
except Exception as exc:
    # N?u eager b? t? ch?i ? m?t version kh?c, th? load kh?ng truy?n attention implementation.
    if 'attn' not in str(exc).lower() and 'attention' not in str(exc).lower():
        raise
    load_kwargs.pop('_attn_implementation', None)
    model = AutoModel.from_pretrained(model_name, **load_kwargs)

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
if torch.cuda.is_available():
    model = model.eval().cuda().to(dtype)
    free, total = torch.cuda.mem_get_info(0)
    print(f'gpu memory after load: free={free/1024**3:.2f}GB total={total/1024**3:.2f}GB dtype={dtype}')
else:
    model = model.eval()

for record in records:
    page_no = int(record['page_number'])
    image_file = str(record['path'])
    result_path = out_dir / f'page_{page_no:03d}.json'
    if result_path.exists() and not force:
        print('skip existing page', page_no)
        continue
    try:
        try:
            result = model.infer(
                tokenizer,
                prompt=prompt,
                image_file=image_file,
                output_path=str(out_dir),
                base_size=base_size,
                image_size=image_size,
                crop_mode=crop_mode,
                save_results=False,
                test_compress=test_compress,
            )
        except TypeError:
            result = model.infer(
                tokenizer,
                prompt=prompt,
                image_file=image_file,
                output_path=str(out_dir),
                base_size=base_size,
                image_size=image_size,
                crop_mode=crop_mode,
                save_results=True,
                test_compress=test_compress,
            )
        if isinstance(result, str):
            markdown = result
        elif isinstance(result, dict):
            markdown = result.get('text') or result.get('markdown') or result.get('content') or json.dumps(result, ensure_ascii=False)
        else:
            markdown = str(result)
        payload = {'page_number': page_no, 'page_image_path': image_file, 'markdown': markdown, 'engine': 'deepseek-ocr'}
    except torch.cuda.OutOfMemoryError as exc:
        payload = {
            'page_number': page_no,
            'page_image_path': image_file,
            'markdown': '',
            'engine': 'deepseek-ocr',
            'error': 'CUDA OutOfMemoryError: ' + str(exc),
            'traceback': traceback.format_exc(),
        }
        result_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
        print('OOM on page', page_no)
        raise
    except Exception as exc:
        payload = {
            'page_number': page_no,
            'page_image_path': image_file,
            'markdown': '',
            'engine': 'deepseek-ocr',
            'error': str(exc),
            'traceback': traceback.format_exc(),
        }
    result_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
    print('done page', page_no, 'chars=', len(payload.get('markdown') or ''), 'error=', payload.get('error'))
    sys.stdout.flush()
'''
WORKER_PATH.write_text(worker_py, encoding='utf-8')
print(WORKER_PATH)


In [ ]:
def selected_gpu_ids():
    if GPU_IDS_TO_USE == 'auto':
        ids = GPU_IDS[:]
    else:
        ids = [int(value.strip()) for value in GPU_IDS_TO_USE.split(',') if value.strip()]
    ids = ids[:max(1, MAX_PARALLEL_GPUS)]
    return ids


def write_jsonl(path, records):
    path.write_text('\n'.join(json.dumps(record, ensure_ascii=False) for record in records) + ('\n' if records else ''), encoding='utf-8')


def run_deepseek():
    gpu_ids = selected_gpu_ids()
    if not gpu_ids and not ALLOW_CPU:
        raise RuntimeError('No GPU selected for DeepSeek-OCR.')
    worker_count = len(gpu_ids) if gpu_ids else 1
    shards = [[] for _ in range(worker_count)]
    for idx, page in enumerate(page_images):
        page_record = {'page_number': int(page['page_number']), 'path': str(Path(page['path']).resolve())}
        shards[idx % worker_count].append(page_record)

    processes = []
    for shard_idx, shard in enumerate(shards):
        if not shard:
            continue
        manifest = MANIFEST_DIR / f'deepseek_shard_{shard_idx}.jsonl'
        write_jsonl(manifest, shard)
        env = os.environ.copy()
        env['USE_TF'] = '0'
        env['TRANSFORMERS_NO_TF'] = '1'
        env['USE_FLAX'] = '0'
        env['TRANSFORMERS_NO_FLAX'] = '1'
        env['TF_CPP_MIN_LOG_LEVEL'] = '3'
        env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
        env['DEEPSEEK_MODEL'] = DEEPSEEK_MODEL
        env['DEEPSEEK_PROMPT'] = DEEPSEEK_PROMPT
        env['DEEPSEEK_ATTN_IMPL'] = DEEPSEEK_ATTN_IMPL
        env['DEEPSEEK_BASE_SIZE'] = str(DEEPSEEK_BASE_SIZE)
        env['DEEPSEEK_IMAGE_SIZE'] = str(DEEPSEEK_IMAGE_SIZE)
        env['DEEPSEEK_CROP_MODE'] = '1' if DEEPSEEK_CROP_MODE else '0'
        env['DEEPSEEK_TEST_COMPRESS'] = '1' if DEEPSEEK_TEST_COMPRESS else '0'
        env['FORCE_DEEPSEEK'] = '1' if FORCE_DEEPSEEK else '0'
        if gpu_ids:
            env['CUDA_VISIBLE_DEVICES'] = str(gpu_ids[shard_idx])
        cmd = [sys.executable, str(WORKER_PATH), str(manifest), str(DEEPSEEK_OUT_DIR)]
        print('launch shard', shard_idx, 'gpu=', gpu_ids[shard_idx] if gpu_ids else 'cpu', 'pages=', len(shard))
        processes.append(subprocess.Popen(cmd, env=env))

    ok = True
    for process in processes:
        code = process.wait()
        ok = ok and (code == 0)
        print('worker exit =', code)
    return ok


deepseek_ok = run_deepseek()
print('deepseek_ok =', deepseek_ok)
if not deepseek_ok:
    raise RuntimeError('DeepSeek worker failed. Xem traceback ? cell output ho?c page_XXX.json trong _deepseek_pages.')


In [ ]:
def clean_text(value):
    return re.sub(r'\s+', ' ', str(value or '').replace('\u00a0', ' ')).strip()


def markdown_text_only(text):
    lines = []
    for line in str(text).splitlines():
        stripped = line.strip()
        if not stripped or stripped == '</break>' or stripped.startswith('!['):
            continue
        stripped = re.sub(r'^#{1,6}\s*', '', stripped)
        lines.append(stripped)
    return clean_text(' '.join(lines))


def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    text = clean_text(text)
    if not text:
        return []
    chunks = []
    start = 0
    while start < len(text):
        end = min(len(text), start + chunk_size)
        if end < len(text):
            punct = max(text.rfind('.', start, end), text.rfind('?', start, end), text.rfind('!', start, end))
            if punct > start + chunk_size * 0.55:
                end = punct + 1
        chunks.append(text[start:end].strip())
        if end >= len(text):
            break
        start = max(0, end - overlap)
    return [chunk for chunk in chunks if chunk]


def load_page_outputs():
    pages = {}
    for path in sorted(DEEPSEEK_OUT_DIR.glob('page_*.json')):
        payload = json.loads(path.read_text(encoding='utf-8'))
        pages[int(payload['page_number'])] = payload
    return pages


deepseek_pages = load_page_outputs()
book_lines = [
    f'# {INPUT_PDF.stem}',
    '',
    f'> Source PDF: `{INPUT_PDF}`',
    f'> Generated at: `{datetime.now(timezone.utc).isoformat()}`',
    f'> Engine: `DeepSeek-OCR`',
    f'> Prompt: `{DEEPSEEK_PROMPT}`',
    '',
]
page_records = []
block_records = []
rag_records = []
errors = []

for page in page_images:
    page_no = int(page['page_number'])
    payload = deepseek_pages.get(page_no, {})
    markdown = str(payload.get('markdown') or '').strip()
    error = payload.get('error')
    if error:
        errors.append({'page_number': page_no, 'error': error})
    if not markdown:
        markdown = f'[OCR missing for PDF page {page_no}]'

    book_lines += [f'## PDF Page {page_no}', '', markdown, '', '</break>', '']
    text = markdown_text_only(markdown)
    block = {'type': 'text', 'order': 0, 'page': page_no, 'page_number': page_no, 'text': text, 'source': 'deepseek-ocr'}
    page_record = {
        'source_pdf': str(INPUT_PDF),
        'page_number': page_no,
        'page_image_path': str(page['path']),
        'text': text,
        'markdown': markdown,
        'text_blocks': [block],
        'engine': 'deepseek-ocr',
        'error': error,
    }
    page_records.append(page_record)
    block_records.append(block)
    (METADATA_DIR / 'pages').mkdir(parents=True, exist_ok=True)
    (METADATA_DIR / 'pages' / f'page_{page_no:03d}.json').write_text(json.dumps(page_record, ensure_ascii=False, indent=2), encoding='utf-8')
    for idx, chunk in enumerate(chunk_text(text), start=1):
        rag_records.append({
            'chunk_id': f'page_{page_no:03d}_{idx:02d}',
            'source_pdf': str(INPUT_PDF),
            'page_number': page_no,
            'text': chunk,
            'images': [],
            'engine': 'deepseek-ocr',
        })

(OUTPUT_DIR / 'book.md').write_text('\n'.join(book_lines).strip() + '\n', encoding='utf-8')
(METADATA_DIR / 'blocks.jsonl').write_text('\n'.join(json.dumps(x, ensure_ascii=False) for x in block_records) + '\n', encoding='utf-8')
(METADATA_DIR / 'images.json').write_text('[]\n', encoding='utf-8')
(OUTPUT_DIR / 'rag_chunks.jsonl').write_text('\n'.join(json.dumps(x, ensure_ascii=False) for x in rag_records) + ('\n' if rag_records else ''), encoding='utf-8')
summary = {
    'source_pdf': str(INPUT_PDF),
    'generated_at': datetime.now(timezone.utc).isoformat(),
    'page_range': {'start': START_PAGE, 'end': END_PAGE},
    'pages_processed': len(page_records),
    'engine': {'ocr': 'deepseek-ocr', 'layout': 'none'},
    'deepseek': {
        'model': DEEPSEEK_MODEL,
        'prompt': DEEPSEEK_PROMPT,
        'attn_impl': DEEPSEEK_ATTN_IMPL,
        'base_size': DEEPSEEK_BASE_SIZE,
        'image_size': DEEPSEEK_IMAGE_SIZE,
        'crop_mode': DEEPSEEK_CROP_MODE,
    },
    'stats': {
        'pages_with_output': sum(1 for p in page_records if p['text'] and not str(p['text']).startswith('[OCR missing')),
        'errors': len(errors),
        'rag_chunks': len(rag_records),
    },
    'errors': errors[:20],
    'outputs': {'markdown': 'book.md', 'rag_chunks': 'rag_chunks.jsonl', 'page_metadata_dir': 'metadata/pages', 'block_metadata': 'metadata/blocks.jsonl', 'image_metadata': 'metadata/images.json'},
}
(METADATA_DIR / 'book.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print('merged:', OUTPUT_DIR)
print('pages:', len(page_records), 'chunks:', len(rag_records), 'errors:', len(errors))
if errors:
    print(errors[:3])


In [ ]:
from docx import Document


def make_docx():
    book_md = OUTPUT_DIR / 'book.md'
    if not book_md.exists():
        return False
    doc = Document()
    doc.add_heading(INPUT_PDF.stem, level=1)
    for line in book_md.read_text(encoding='utf-8').splitlines():
        stripped = line.strip()
        if not stripped or stripped.startswith('> ') or stripped == '</break>':
            continue
        if stripped.startswith('# '):
            continue
        if stripped.startswith('## '):
            doc.add_heading(stripped[3:].strip(), level=2)
        elif stripped.startswith('### '):
            doc.add_heading(stripped[4:].strip(), level=3)
        elif stripped.startswith('#### '):
            doc.add_heading(stripped[5:].strip(), level=4)
        elif stripped.startswith('!['):
            doc.add_paragraph(stripped)
        else:
            doc.add_paragraph(stripped)
    doc.save(str(OUTPUT_DIR / 'book.docx'))
    return True

print('docx:', make_docx())
for path in [OUTPUT_DIR / 'book.md', OUTPUT_DIR / 'book.docx', OUTPUT_DIR / 'rag_chunks.jsonl', METADATA_DIR / 'book.json']:
    if path.exists():
        print(path, path.stat().st_size)


In [ ]:
zip_path = OUTPUT_DIR.with_suffix('.zip')
if ZIP_OUTPUT:
    if zip_path.exists():
        zip_path.unlink()
    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as archive:
        for path in sorted(OUTPUT_DIR.rglob('*')):
            if path.is_file():
                archive.write(path, path.relative_to(OUTPUT_DIR.parent))
    print('zip:', zip_path, zip_path.stat().st_size)

print('\nSanity check:')
summary = json.loads((METADATA_DIR / 'book.json').read_text(encoding='utf-8'))
print(json.dumps(summary, ensure_ascii=False, indent=2)[:2000])

print('\nPreview book.md:')
print((OUTPUT_DIR / 'book.md').read_text(encoding='utf-8')[:2500])

try:
    from IPython.display import display, Image as IPImage
    first_page = page_images[0]['path'] if page_images else None
    if first_page:
        display(IPImage(filename=str(first_page)))
except Exception as exc:
    print(exc)

print('\nImportant files:')
for rel in ['book.md', 'book.docx', 'rag_chunks.jsonl', 'metadata/book.json', 'metadata/images.json']:
    path = OUTPUT_DIR / rel
    print(rel, 'OK' if path.exists() else 'MISSING', path.stat().st_size if path.exists() else '')
